# 05 — FCI validation

Run full configuration interaction on the (12e, 12o) target active space and
check that the correlation energy missed by Hartree–Fock exceeds the QPE target
precision $\varepsilon = 1.6$ mHa. If $|E_{\text{FCI}} - E_{\text{HF}}| > \varepsilon$,
QPE at chemical accuracy resolves a meaningful energy gap; otherwise the
quantum estimate would not improve on the mean-field reference.

In [1]:
import json
import os
import sys
import numpy as np
from pyscf import fci

sys.path.insert(0, ".")
from config import load_config, ensure_dirs
cfg = load_config()
ensure_dirs(cfg)

npz_path = os.path.join(cfg["paths"]["data_dir"], "integrals.npz")
npz = np.load(npz_path)
ehf_total = float(npz["ehf"])

target = {
    "nelec": int(npz["target_nelec"]),
    "norb": int(npz["target_norb"]),
    "e_core": float(npz["target_e_core"]),
    "h1e_eff": npz["target_h1e_eff"],
    "h2e_phys": npz["target_h2e_phys"],
}
epsilon = cfg["qpe"]["epsilon_ha"]

## FCI on the (12e, 12o) target space

In [2]:
h2e_chem = target["h2e_phys"].transpose(0, 2, 1, 3)

cisolver = fci.direct_spin0.FCI()
e_fci_active, _ = cisolver.kernel(
    target["h1e_eff"], h2e_chem, target["norb"], target["nelec"]
)
e_fci_total = target["e_core"] + e_fci_active

# HF energy in the active space directly from integrals
occ = target["nelec"] // 2
h1 = target["h1e_eff"]
h2 = h2e_chem
e_hf_active = 0.0
for i in range(occ):
    e_hf_active += 2.0 * h1[i, i]
    for j in range(occ):
        e_hf_active += 2.0 * h2[i, i, j, j] - h2[i, j, j, i]
e_hf_total_active = target["e_core"] + e_hf_active

correlation_ha = e_fci_active - e_hf_active

print(f"Core energy:                {target['e_core']:>14.6f} Ha")
print(f"HF energy (active space):   {e_hf_active:>14.10f} Ha")
print(f"FCI energy (active space):  {e_fci_active:>14.10f} Ha")
print(f"HF total (core + active):   {e_hf_total_active:>14.10f} Ha")
print(f"FCI total (core + active):  {e_fci_total:>14.10f} Ha")
print()
print(f"Correlation energy:         {correlation_ha:.6f} Ha "
      f"({correlation_ha * 27.2114:.4f} eV)")
print(f"QPE target precision:       {epsilon:.4f} Ha "
      f"({epsilon * 27.2114:.4f} eV)")

Core energy:                  -2498.518333 Ha
HF energy (active space):   -26.4155708242 Ha
FCI energy (active space):  -26.4552636422 Ha
HF total (core + active):   -2524.9339034726 Ha
FCI total (core + active):  -2524.9735962906 Ha

Correlation energy:         -0.039693 Ha (-1.0801 eV)
QPE target precision:       0.0016 Ha (0.0435 eV)


## Validation verdict

In [3]:
validated = abs(correlation_ha) > epsilon

if validated:
    print(f"Validated: |E_corr| = {abs(correlation_ha) * 1000:.2f} mHa "
          f"exceeds QPE precision (1.6 mHa).")
    print("QPE at chemical accuracy resolves correlation energy missed by HF.")
else:
    print("Warning: correlation energy is smaller than QPE precision.")
    print("QPE would not resolve a meaningful energy difference here.")

out_path = os.path.join(cfg["paths"]["data_dir"], "fci_validation.json")
with open(out_path, "w") as f:
    json.dump({
        "core_energy_ha": target["e_core"],
        "hf_active_ha": e_hf_active,
        "fci_active_ha": e_fci_active,
        "hf_total_ha": e_hf_total_active,
        "fci_total_ha": e_fci_total,
        "correlation_ha": float(correlation_ha),
        "epsilon_ha": epsilon,
        "validated": bool(validated),
    }, f, indent=2)
print(f"saved {out_path}")

Validated: |E_corr| = 39.69 mHa exceeds QPE precision (1.6 mHa).
QPE at chemical accuracy resolves correlation energy missed by HF.
saved data/fci_validation.json
